# Análisis geoespacial en manufactura con Python
## Un caso ficticio de abastecimiento, riesgo territorial y ubicación de un almacén

**Nivel:** principiante. **Modalidad:** tutorial ejecutable en Google Colab, con CPU.

### Qué aprenderá

Convertirá una tabla de proveedores en información geográfica, medirá distancias correctamente, dibujará áreas de cobertura, identificará proveedores dentro de una zona ficticia de riesgo y comparará tres ubicaciones de almacén. Cada bloque incluye su propósito, una explicación del código
y una guía para interpretar la salida. Los resultados numéricos y las conclusiones se generan con los datos calculados.

### Cómo correrlo en Colab

1. Abra [Google Colab](https://colab.research.google.com/) y cargue este archivo `.ipynb` mediante **Archivo → Subir notebook**.
2. Utilice un entorno de **Python 3 con CPU**; no necesita GPU, Drive, contraseñas ni archivos de entrada.
3. Ejecute **Entorno de ejecución → Ejecutar todas**. La primera celda instala las dependencias que falten; requiere Internet.
4. Si trabaja en una sesión donde ya importó otras versiones y Colab solicita reiniciarla, reinicie y vuelva a ejecutar todo.
5. Revise tablas, mapas e interpretaciones en orden. Al final puede descargar un ZIP con los resultados.

Los cálculos y mapas estáticos no consultan datos externos. El mapa interactivo utiliza JavaScript y un fondo cartográfico de Internet; si ese fondo no carga, los mapas estáticos conservan la información analítica.
Los archivos de salida se guardan en `salidas_geoespacial`, dentro del entorno donde ejecute el notebook.

## 1. Contexto: ¿qué problema de manufactura resolvemos?

**MetalNorte Componentes** es una empresa inventada que fabrica soportes y carcasas metálicas.
Su planta se ubica, solo para este ejercicio, en el entorno geográfico de Monterrey, México.
Recibe acero, empaques y piezas semiacabadas de 60 proveedores distribuidos en tres grupos territoriales.
En el escenario actual, cada proveedor transporta su material directamente a la planta.

La gerencia observa retrasos, pero una tabla de compras no le permite ver si varios proveedores
comparten una misma exposición territorial. Además, estudia un almacén de consolidación:
los proveedores entregarían allí y el material viajaría agrupado hacia la planta. Agrupar material
podría reducir la tarifa por tonelada-kilómetro del último tramo, aunque también introduce manejo,
costo fijo y posibles recorridos adicionales.

**Pregunta de decisión:** ¿qué revela la ubicación de los proveedores y cuál de tres almacenes
candidatos merece una evaluación operativa más profunda frente a mantener el abastecimiento directo?

Se responderán cuatro preguntas concretas:

- ¿Qué proporción de proveedores y de toneladas queda a menos de 25 km de la planta?
- ¿Cuántos proveedores, toneladas y entregas se ubican en una zona de riesgo ficticia?
- ¿Qué candidato queda más cerca de cada proveedor y qué volumen concentraría esa asignación?
- ¿Qué costo mensual simulado tendría pasar todo el flujo por cada candidato, respetando su capacidad?

**Alcance:** evaluación preliminar de una red de abastecimiento. No se optimizan rutas de camiones,
secuencias de visitas, inventarios ni niveles de servicio. Tampoco se estima un efecto causal sobre retrasos.

## 2. La técnica geoespacial, explicada paso a paso

### Una tabla que también sabe dónde están las cosas

El análisis geoespacial combina atributos —por ejemplo, toneladas o entregas— con una ubicación.
Una tabla convencional puede decir que un proveedor entrega 30 toneladas. Una tabla geográfica
también permite preguntar a qué distancia está, en qué zona cae y qué instalaciones tiene cerca.
No es un algoritmo único ni exige inteligencia artificial predictiva: aquí utilizamos operaciones
geométricas para responder preguntas de manufactura.

Trabajaremos con **datos vectoriales**. Un **punto** representa una instalación; un **polígono**,
una superficie como una zona de cobertura; una **línea** podría representar una carretera si
dispusiéramos de su trazado. Los datos ráster, como imágenes satelitales o cuadrículas de elevación,
pertenecen también al análisis geoespacial, pero no son necesarios en este ejemplo.

GeoPandas amplía las tablas de pandas con una columna `geometry`. Esa columna guarda puntos o
polígonos y permite ejecutar operaciones espaciales junto a filtros y agrupaciones tabulares.
Es una biblioteca mantenida; fijamos la versión 1.1.4 para que el tutorial sea reproducible, en vez
de depender de una versión futura sin probar. [Proyecto y versión publicada](https://pypi.org/project/geopandas/).

### Coordenadas y sistemas de referencia: elegir la regla correcta

La **longitud** indica la posición este–oeste y la **latitud**, norte–sur. En GeoPandas creamos puntos
en orden `(longitud, latitud)`, equivalente a `(x, y)`. Folium recibe sus posiciones en el orden
`[latitud, longitud]`; esta diferencia se señalará al construir el mapa.

Un **CRS**, o sistema de referencia de coordenadas, indica cómo interpretar esos números.
EPSG:4326 almacena grados geográficos. Para medir en una región alrededor de Monterrey, usaremos
EPSG:32614, WGS 84 / UTM zona 14 norte, cuyas unidades son metros. `set_crs` declara qué significan
coordenadas existentes; `to_crs` transforma las coordenadas a otro sistema. Cambiar solamente la
etiqueta no convierte grados en metros. [Guía de proyecciones de GeoPandas](https://geopandas.org/en/stable/docs/user_guide/projections.html).

Imagine que dos puntos están separados por 0.1 grados: esa cantidad no significa 0.1 kilómetros.
Después de proyectar, podemos medir una distancia en metros y dividir entre 1,000. La proyección
introduce una distorsión pequeña a esta escala local; no debe reutilizarse automáticamente para
instalaciones repartidas por todo un continente.

### Las cuatro operaciones que utilizaremos

| Operación | Idea sencilla | Pregunta de manufactura |
|---|---|---|
| Distancia | Medir separación entre puntos | ¿Qué tan lejos está cada proveedor de la planta? |
| Buffer | Dibujar un área alrededor de un punto con un radio dado | ¿Qué proveedores están dentro de 25 km? |
| Unión espacial | Unir tablas porque sus figuras se intersectan | ¿Qué proveedores están dentro de una zona? |
| Vecino más cercano | Buscar la instalación de menor distancia | ¿Cuál candidato está más cerca de cada proveedor? |

Un buffer es un círculo aproximado mediante muchos segmentos. Sirve como referencia de proximidad;
no incorpora carreteras, puentes, montañas ni tráfico. Para indicadores exactos de cobertura en
este modelo usaremos la condición `distancia <= radio`; el polígono será su representación visual.
[Documentación de buffer](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.buffer.html).

Una unión espacial con `intersects` relaciona puntos con polígonos, incluyendo la frontera.
Un proveedor en dos zonas podría aparecer dos veces; por eso verificamos la cantidad de filas.
Aquí solo se crea una zona ficticia. [Documentación de sjoin](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin.html).

El vecino más cercano tampoco es automáticamente el mejor proveedor o almacén: no considera
capacidad, precio ni confiabilidad. Si hay empate, GeoPandas puede devolver más de una fila;
aplicaremos un desempate por identificador para conservar un registro por proveedor.
[Documentación de sjoin_nearest](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin_nearest.html).

### Cómo resumir sin perder información

La distancia media simple da el mismo peso a cada proveedor. La distancia ponderada por volumen
da mayor importancia a quienes transportan más material:

$$\bar d_w = \frac{\sum_i toneladas_i\,distancia_i}{\sum_i toneladas_i}.$$

Por ejemplo, si un proveedor de 10 toneladas está a 5 km y otro de 90 toneladas a 25 km,
la media simple es 15 km; la ponderada es 23 km. Esta última describe mejor dónde se concentra
el esfuerzo de transporte por tonelada. Ninguna es un tiempo de traslado.

## 3. Preparación
### Bloque 1 · Instalar las dependencias

`importlib.metadata.version` consulta las versiones instaladas sin importar aún las librerías.
La lista `requisitos` fija GeoPandas y Folium, y admite versiones compatibles de las librerías auxiliares.
`Requirement` interpreta esas reglas: `==` exige una versión y los rangos aceptan varias.

El ciclo revisa cada requisito y acumula solo los faltantes o incompatibles. `subprocess.check_call`
ejecuta pip con el mismo Python del notebook, identificado por `sys.executable`; esto evita instalar
en otro entorno por accidente. Si pip falla, la celda se detiene y muestra el error.
Esta instalación ocurre antes de importar NumPy, pandas o GeoPandas. Python 3.10 o posterior es necesario.

In [ ]:
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
from packaging.requirements import Requirement

requisitos = [
    "geopandas==1.1.4", "folium==0.20.0",
    "numpy>=1.26,<3", "pandas>=2.2,<3", "matplotlib>=3.8,<4",
]
pendientes = []
for texto in requisitos:
    requisito = Requirement(texto)
    try:
        cumple = version(requisito.name) in requisito.specifier
    except PackageNotFoundError:
        cumple = False
    if not cumple:
        pendientes.append(texto)
if pendientes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pendientes])
print("Dependencias preparadas. Continúe con las importaciones.")

**Cómo leer la salida.** El mensaje final significa que las versiones solicitadas están disponibles. No es un resultado
de manufactura. En una segunda ejecución normalmente no se descargará nada. Si aparece un error
de conexión, vuelva a ejecutar esta celda cuando Colab tenga acceso a Internet.

### Bloque 2 · Importar herramientas y registrar versiones

`numpy` genera datos y hace cálculos; `pandas` maneja tablas; `geopandas` agrega geometrías.
`box` construye un rectángulo geográfico con Shapely. `matplotlib` dibuja figuras estáticas y
`folium` prepara un mapa con controles y ventanas informativas. `display` y `Markdown` presentan
tablas y explicaciones calculadas dentro de Colab.

`Path` construye rutas portables; la carpeta se crea con `exist_ok=True` para poder volver a ejecutar
sin errores. Guardamos las versiones efectivas, porque las dependencias auxiliares pueden variar.
El formato numérico solo cambia la presentación; los cálculos conservan su precisión.

In [ ]:
import json
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from shapely.geometry import box
import folium
from IPython.display import display, Markdown

SALIDA = Path("salidas_geoespacial")
SALIDA.mkdir(exist_ok=True)
pd.options.display.float_format = "{:,.2f}".format
plt.rcParams.update({"figure.dpi": 115, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})
versiones = {p: version(p) for p in
            ["geopandas", "shapely", "pyproj", "folium", "numpy", "pandas", "matplotlib"]}
display(pd.Series(versiones, name="Versión instalada").to_frame())

**Cómo leer la salida.** La tabla documenta el entorno real de ejecución. GeoPandas debe mostrar 1.1.4. Las versiones
se exportarán junto a los resultados para facilitar que otra persona reproduzca el ejercicio.

### Bloque 3 · Definir supuestos visibles

Centralizamos parámetros para que el caso sea fácil de modificar. La semilla fija el punto de
partida del generador aleatorio: con el mismo código y entorno se obtienen los mismos datos.
El radio expresa cercanía operativa, no una promesa de entrega. `FACTOR_RECORRIDO` multiplica
distancias rectas por 1.25 como aproximación didáctica, sin consultar una red vial.

Las tarifas son supuestos inventados en MXN por tonelada-kilómetro. Se supone que consolidar
permite un transporte más barato hacia la planta; el ahorro no está garantizado, porque debemos
sumar recolección, manejo y costo fijo. Cada tonelada se contabiliza una vez en cada tramo.
No se modelan viajes vacíos, mínimos por camión, impuestos, inversión inicial ni inventario.

In [ ]:
SEMILLA = 42
PERIODO = "2026-08"
CRS_GEOGRAFICO = "EPSG:4326"
CRS_METRICO = "EPSG:32614"
RADIO_KM = 25.0
FACTOR_RECORRIDO = 1.25
TARIFA_DIRECTA = 7.0
TARIFA_RECOLECCION = 5.0
TARIFA_CONSOLIDADA = 1.6
MANEJO_MXN_T = 12.0
rng = np.random.default_rng(SEMILLA)

display(pd.DataFrame({
    "Supuesto": ["Radio de cercanía", "Factor de recorrido", "Transporte directo",
                 "Proveedor a almacén", "Almacén a planta", "Manejo en almacén"],
    "Valor": [RADIO_KM, FACTOR_RECORRIDO, TARIFA_DIRECTA,
              TARIFA_RECOLECCION, TARIFA_CONSOLIDADA, MANEJO_MXN_T],
    "Unidad": ["km rectos", "multiplicador", "MXN/t-km", "MXN/t-km", "MXN/t-km", "MXN/t"],
}))

**Cómo leer la salida.** Estos números son entradas del modelo, no hallazgos. Una tarifa consolidada menor favorece
el almacén; costos de manejo elevados pueden eliminar esa ventaja. Más adelante variaremos dos
supuestos para observar si la decisión depende demasiado de ellos.

## 4. Datos ficticios
### Bloque 4 · Crear planta, candidatos y proveedores

La tabla de planta contiene una instalación. La tabla de candidatos incluye tres ubicaciones,
su capacidad mensual y su costo fijo mensual. Las coordenadas están dentro de un entorno real
para facilitar el mapa, pero no identifican instalaciones reales ni predios disponibles.

`np.repeat` construye tres grupos de 26, 22 y 12 proveedores. Partimos de un centro por grupo y
agregamos una pequeña desviación aleatoria a longitud y latitud. Esto produce concentraciones
territoriales fáciles de estudiar. `rng.integers(8, 46)` genera toneladas enteras entre 8 y 45.
La unidad de cada fila es **un proveedor en un mes completo**; sus toneladas son el flujo mensual,
no el peso de un embarque individual. Guardamos las tablas de entrada antes de transformarlas.

In [ ]:
planta_df = pd.DataFrame({"planta": ["MetalNorte"], "longitud": [-100.30], "latitud": [25.67]})
candidatos_df = pd.DataFrame({
    "candidato": ["A_Norte", "B_Este", "C_Oeste"],
    "longitud": [-100.30, -100.08, -100.53],
    "latitud": [25.87, 25.75, 25.71],
    "capacidad_t_mes": [2500, 2500, 1300],
    "fijo_mxn_mes": [18000, 14000, 11000],
})
grupos = np.repeat(["Norte", "Este", "Oeste"], [26, 22, 12])
centros = {"Norte": (-100.30, 25.94), "Este": (-99.99, 25.78), "Oeste": (-100.57, 25.72)}
proveedores_df = pd.DataFrame({
    "proveedor": [f"P{i:03d}" for i in range(1, len(grupos) + 1)],
    "periodo": PERIODO,
    "grupo_origen": grupos,
    "longitud": [centros[g][0] for g in grupos] + rng.normal(0, 0.045, len(grupos)),
    "latitud": [centros[g][1] for g in grupos] + rng.normal(0, 0.035, len(grupos)),
    "toneladas_mes": rng.integers(8, 46, len(grupos)),
})
for nombre, tabla in [("proveedores_entrada", proveedores_df), ("planta_entrada", planta_df),
                       ("candidatos_entrada", candidatos_df)]:
    tabla.to_csv(SALIDA / f"{nombre}.csv", index=False)
display(proveedores_df.head(8))
display(candidatos_df)

**Cómo leer la salida.** La vista muestra solo ocho proveedores, pero el análisis utiliza los 60. `grupo_origen` documenta
cómo se generaron; no es una clasificación descubierta por un algoritmo. Las capacidades distintas
permiten comprobar que una alternativa barata puede ser insuficiente para todo el volumen.

### Bloque 5 · Comprobar los datos antes de mapear

Las instrucciones `assert` detienen el análisis si no se cumple una condición necesaria.
Comprobamos identificadores únicos, ausencia de valores vacíos, cantidades positivas y coordenadas
dentro de un rango local razonable. El rango detecta, por ejemplo, latitud y longitud intercambiadas.

Después agregamos proveedores y toneladas por grupo. Una suma de control verifica que todos
los grupos recuperan el volumen original. Estas comprobaciones ayudan a descubrir errores
de estructura; no prueban que una dirección real esté correctamente geocodificada.

In [ ]:
assert len(proveedores_df) == 60
assert proveedores_df["proveedor"].is_unique
assert candidatos_df["candidato"].is_unique
for tabla in [proveedores_df, planta_df, candidatos_df]:
    assert not tabla.isna().any().any()
    assert tabla["longitud"].between(-101, -99).all()
    assert tabla["latitud"].between(25, 27).all()
assert proveedores_df["toneladas_mes"].gt(0).all()
assert candidatos_df["capacidad_t_mes"].gt(0).all()
resumen_origen = proveedores_df.groupby("grupo_origen").agg(
    proveedores=("proveedor", "size"), toneladas=("toneladas_mes", "sum"))
TOTAL_T = float(proveedores_df["toneladas_mes"].sum())
assert resumen_origen["toneladas"].sum() == TOTAL_T
display(resumen_origen)
print(f"Datos válidos: 60 proveedores y {TOTAL_T:,.0f} toneladas en {PERIODO}.")

**Cómo leer la salida.** Los conteos deben sumar 60 y las toneladas deben coincidir con el total impreso.
Una mayor concentración de volumen en un grupo puede influir más en la decisión que el número
de puntos en el mapa. Por eso reportamos ambas medidas.

## 5. Análisis espacial
### Bloque 6 · Crear geometrías y convertir grados a metros

La función `convertir_a_geo` evita repetir el mismo procedimiento para tres tablas.
`gpd.points_from_xy` crea un punto con longitud como x y latitud como y; `crs=EPSG:4326`
indica que esos números son grados. La función devuelve un `GeoDataFrame`, es decir,
una tabla con una columna de geometría activa.

Creamos copias geográficas para el mapa interactivo y copias proyectadas con `to_crs` para
calcular distancias. Las comprobaciones confirman que el CRS es proyectado, usa metros y contiene
geometrías válidas y no vacías. Todas las capas de un cálculo deben compartir el mismo CRS.

In [ ]:
def convertir_a_geo(tabla):
    return gpd.GeoDataFrame(
        tabla.copy(), geometry=gpd.points_from_xy(tabla["longitud"], tabla["latitud"]),
        crs=CRS_GEOGRAFICO)

proveedores_geo = convertir_a_geo(proveedores_df)
planta_geo = convertir_a_geo(planta_df)
candidatos_geo = convertir_a_geo(candidatos_df)
proveedores = proveedores_geo.to_crs(CRS_METRICO)
planta = planta_geo.to_crs(CRS_METRICO)
candidatos = candidatos_geo.to_crs(CRS_METRICO)
for capa in [proveedores, planta, candidatos]:
    assert capa.crs.is_projected
    assert capa.crs.axis_info[0].unit_name == "metre"
    assert capa.geometry.is_valid.all() and not capa.geometry.is_empty.any()
display(proveedores[["proveedor", "geometry"]].head(3))
print("CRS para medir:", proveedores.crs)

**Cómo leer la salida.** Ahora `POINT` muestra valores grandes que representan metros este y norte en UTM.
Los proveedores siguen en el mismo lugar de la Tierra: cambió la forma de expresar su posición.
No interprete las columnas originales `latitud` y `longitud` como coordenadas UTM; la transformación
se encuentra en `geometry`.

### Bloque 7 · Medir cercanía y cobertura actual

Tomamos la geometría de la única planta con `iloc[0]`. `distance` mide la separación entre cada
proveedor y ese punto, en metros; dividir entre 1,000 produce kilómetros rectos. `buffer` genera
el polígono de referencia de 25 km. Para clasificar usamos la distancia numérica, incluyendo
proveedores exactamente en el límite mediante `<=`.

`np.average` calcula la distancia ponderada con toneladas como pesos. Una media de valores
booleanos convierte `True` en 1 y `False` en 0, de modo que expresa la fracción de proveedores
dentro del radio. Para cobertura de volumen sumamos únicamente sus toneladas y dividimos entre
el volumen total. Son denominadores distintos y responden preguntas distintas.

In [ ]:
punto_planta = planta.geometry.iloc[0]
proveedores["distancia_planta_km"] = proveedores.geometry.distance(punto_planta) / 1000
proveedores["dentro_radio"] = proveedores["distancia_planta_km"] <= RADIO_KM
cobertura = gpd.GeoDataFrame({"zona": [f"Radio {RADIO_KM:g} km"]},
    geometry=planta.geometry.buffer(RADIO_KM * 1000, resolution=64), crs=CRS_METRICO)
distancia_media = proveedores["distancia_planta_km"].mean()
distancia_ponderada = np.average(proveedores["distancia_planta_km"], weights=proveedores["toneladas_mes"])
cobertura_proveedores = 100 * proveedores["dentro_radio"].mean()
cobertura_toneladas = 100 * proveedores.loc[proveedores["dentro_radio"], "toneladas_mes"].sum() / TOTAL_T
display(Markdown(f"""
**Diagnóstico de cercanía:** la distancia media simple es **{distancia_media:.1f} km** y la
ponderada por volumen es **{distancia_ponderada:.1f} km**. Dentro de {RADIO_KM:g} km quedan
**{cobertura_proveedores:.1f}% de los proveedores** y **{cobertura_toneladas:.1f}% de las toneladas**.
El porcentaje restante de volumen, **{100-cobertura_toneladas:.1f}%**, proviene de fuera del radio.
Esto describe proximidad; no significa que esas entregas sean tardías.
"""))

**Cómo leer la salida.** Compare cobertura de proveedores con cobertura de toneladas. Si la primera fuera 40% y la segunda
60%, los proveedores cercanos moverían una porción de material mayor que su proporción numérica.
La interpretación impresa utiliza los resultados reales del ejercicio, no esos números ilustrativos.

### Bloque 8 · Identificar exposición a una zona ficticia

`box` crea un rectángulo inventado al este de la planta. Lo etiquetamos como zona de interrupción
simulada; no procede de un mapa de amenazas. Primero declaramos sus coordenadas geográficas y
después lo proyectamos al mismo CRS de los proveedores.

`sjoin(..., how="left", predicate="intersects")` conserva todos los proveedores y agrega el nombre
de zona cuando el punto cae dentro o en el borde. `notna` transforma esa coincidencia en un indicador.
Eliminamos `index_right`, un identificador técnico de la tabla de polígonos. Revisamos unicidad
y cantidad de filas para impedir que una unión duplique toneladas silenciosamente.

In [ ]:
zona = gpd.GeoDataFrame({"zona_riesgo": ["Interrupción simulada del sector este"]},
    geometry=[box(-100.09, 25.71, -99.88, 25.89)], crs=CRS_GEOGRAFICO).to_crs(CRS_METRICO)
proveedores = proveedores.sjoin(zona, how="left", predicate="intersects")
proveedores["expuesto"] = proveedores["zona_riesgo"].notna()
proveedores = proveedores.drop(columns="index_right").sort_values("proveedor").reset_index(drop=True)
assert len(proveedores) == 60 and proveedores["proveedor"].is_unique
expuestos_n = int(proveedores["expuesto"].sum())
expuestos_t = float(proveedores.loc[proveedores["expuesto"], "toneladas_mes"].sum())
display(Markdown(f"""
**Exposición territorial:** **{expuestos_n} proveedores** se intersectan con la zona ficticia.
Representan **{expuestos_t:,.0f} t/mes**, equivalentes a **{100*expuestos_t/TOTAL_T:.1f}% del volumen**.
Este porcentaje mide material asociado a ubicaciones dentro del polígono; no es una probabilidad
de interrupción ni una predicción de toneladas perdidas. Proveedores fuera de la zona también
podrían depender de carreteras que la atraviesen, algo que este modelo no observa.
"""))

**Cómo leer la salida.** Un punto dentro del polígono queda marcado como expuesto aunque no tenga retrasos. La exposición
proviene de la geometría, mientras que el desempeño de entregas se simulará por separado.
Así evitamos definir la zona a partir del mismo resultado que queremos comparar.

### Bloque 9 · Simular entregas y comparar tasas de retraso

Creamos un segundo generador con otra semilla para que las entregas no dependan de cuántos números
aleatorios consumió la creación de coordenadas. Cada proveedor tiene entre 12 y 35 entregas en el mes.
La probabilidad inventada de retraso aumenta con la distancia y con la exposición. `clip` impide
probabilidades fuera del intervalo definido; `binomial` produce un número de retrasos entre cero
y el total de entregas de cada proveedor.

Agrupamos por exposición y calculamos `retrasos totales / entregas totales`. No promediamos las tasas
individuales, pues proveedores con distinto número de entregas necesitan pesos diferentes.
El patrón de mayor riesgo se introduce deliberadamente en la simulación para aprender la técnica:
no es una relación descubierta en observaciones reales ni evidencia de causalidad.

In [ ]:
rng_entregas = np.random.default_rng(SEMILLA + 1)
proveedores["entregas"] = rng_entregas.integers(12, 36, len(proveedores))
probabilidad = np.clip(0.04 + proveedores["distancia_planta_km"] / 500
                      + 0.12 * proveedores["expuesto"].astype(int), 0.02, 0.60)
proveedores["retrasos"] = rng_entregas.binomial(proveedores["entregas"].to_numpy(), probabilidad.to_numpy())
proveedores["tasa_retraso_pct"] = 100 * proveedores["retrasos"] / proveedores["entregas"]
resumen_riesgo = proveedores.groupby("expuesto").agg(
    proveedores=("proveedor", "size"), toneladas=("toneladas_mes", "sum"),
    entregas=("entregas", "sum"), retrasos=("retrasos", "sum"))
resumen_riesgo["tasa_retraso_pct"] = 100 * resumen_riesgo["retrasos"] / resumen_riesgo["entregas"]
resumen_riesgo.index = resumen_riesgo.index.map({False: "Fuera de zona", True: "Dentro de zona"})
tasa_global = 100 * proveedores["retrasos"].sum() / proveedores["entregas"].sum()
display(resumen_riesgo)
display(Markdown(f"""La tasa global simulada de retraso es **{tasa_global:.1f}%**. Cada tasa de la tabla usa
como denominador las entregas de su propio grupo; las toneladas no son el denominador de esa tasa."""))

**Cómo leer la salida.** Observe tanto el porcentaje como el número de entregas que lo sostiene. Un proveedor con pocas
entregas puede mostrar una tasa alta por pocos eventos. Una diferencia entre grupos puede reflejar
distancia, contratos, tipo de insumo u otros factores; aquí refleja además las reglas de simulación.
No calculamos intervalos de confianza ni afirmamos significancia estadística.

### Bloque 10 · Dibujar el mapa analítico y comparar retrasos

La figura separa dos preguntas: ubicación a la izquierda y tasas por exposición a la derecha.
Las capas se dibujan en orden: área de riesgo, borde de cobertura, proveedores, planta y candidatos.
Los puntos rojos indican exposición; las estrellas y triángulos identifican instalaciones.
Las coordenadas de los ejes se muestran en kilómetros UTM y el aspecto igual evita deformar distancias.

En la gráfica de barras el eje comienza en cero; cada etiqueta muestra la tasa agregada del grupo.
`savefig` conserva una imagen independiente del mapa web. `plt.show` la presenta dentro del notebook.
La gráfica es descriptiva y no agrega una prueba estadística.

In [ ]:
fig, (ax_mapa, ax_tasas) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.5, 1]})
zona.plot(ax=ax_mapa, color="#f7c9c7", edgecolor="#cc554d", alpha=0.5)
cobertura.boundary.plot(ax=ax_mapa, color="#218c74", linestyle="--", linewidth=1.7)
for expuesto, color, etiqueta in [(False, "#246a95", "Proveedor fuera de zona"),
                                   (True, "#c83f49", "Proveedor dentro de zona")]:
    proveedores.loc[proveedores["expuesto"] == expuesto].plot(
        ax=ax_mapa, color=color, markersize=35, label=etiqueta)
planta.plot(ax=ax_mapa, marker="*", color="#152536", markersize=220, label="Planta")
candidatos.plot(ax=ax_mapa, marker="^", color="#d68a13", markersize=95, label="Candidato")
for fila in candidatos.itertuples():
    ax_mapa.annotate(fila.candidato, (fila.geometry.x, fila.geometry.y), xytext=(5, 7),
                    textcoords="offset points", fontsize=9)
ax_mapa.set(title="Red ficticia y radio de cercanía de 25 km", xlabel="Este UTM (km)", ylabel="Norte UTM (km)")
ax_mapa.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1000:,.0f}"))
ax_mapa.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y/1000:,.0f}"))
ax_mapa.set_aspect("equal")
ax_mapa.legend(loc="lower left", fontsize=8)
barras = ax_tasas.bar(resumen_riesgo.index, resumen_riesgo["tasa_retraso_pct"], color=["#246a95", "#c83f49"])
ax_tasas.bar_label(barras, fmt="%.1f%%", padding=4)
ax_tasas.set(title=f"Retrasos simulados · {PERIODO}", ylabel="Entregas con retraso (%)",
             ylim=(0, max(35, resumen_riesgo["tasa_retraso_pct"].max() * 1.25)))
ax_tasas.grid(axis="y", alpha=0.15)
fig.suptitle("MetalNorte · Diagnóstico territorial | Datos completamente sintéticos", fontweight="bold")
fig.tight_layout()
fig.savefig(SALIDA / "01_diagnostico_territorial.png", bbox_inches="tight")
plt.show()

**Cómo leer la salida.** El contorno discontinuo muestra cercanía a la planta y el rectángulo representa la zona simulada.
La agrupación de puntos rojos ayuda a ver una exposición compartida que una tabla no hace evidente.
El mapa no contiene carreteras: cualquier recorrido que imagine entre dos puntos todavía necesita
verificarse. La barra más alta identifica una mayor tasa en este conjunto simulado, sin explicar por sí sola su causa.

### Bloque 11 · Encontrar el candidato más cercano a cada proveedor

`sjoin_nearest` busca la menor distancia entre cada proveedor y los tres candidatos; `distance_col`
guarda esa distancia en metros. Si dos candidatos están exactamente a la misma distancia, la función
puede generar dos registros. Ordenamos por proveedor, distancia e identificador, y conservamos
la primera coincidencia. El desempate alfabético es una regla administrativa explícita.

Luego agrupamos las toneladas por candidato y las comparamos con su capacidad. Esta asignación
responde una pregunta de proximidad suponiendo los tres sitios disponibles simultáneamente.
Es un diagnóstico separado del siguiente modelo, que estudia abrir **solo uno** y enviarle todo el flujo.
No alimentaremos el modelo de un almacén con las cargas de este escenario de tres almacenes.

In [ ]:
asignacion = proveedores[["proveedor", "toneladas_mes", "geometry"]].sjoin_nearest(
    candidatos[["candidato", "geometry"]], how="left", distance_col="distancia_m")
empates_extra = len(asignacion) - len(proveedores)
asignacion = (asignacion.sort_values(["proveedor", "distancia_m", "candidato"])
              .drop_duplicates("proveedor").reset_index(drop=True))
asignacion["distancia_candidato_km"] = asignacion["distancia_m"] / 1000
carga_cercana = asignacion.groupby("candidato").agg(
    proveedores=("proveedor", "size"), toneladas=("toneladas_mes", "sum"))
carga_cercana = candidatos_df[["candidato", "capacidad_t_mes"]].merge(
    carga_cercana, on="candidato", how="left", validate="one_to_one").fillna(0)
carga_cercana["utilizacion_pct"] = 100 * carga_cercana["toneladas"] / carga_cercana["capacidad_t_mes"]
assert asignacion["proveedor"].is_unique and len(asignacion) == 60
assert carga_cercana["toneladas"].sum() == TOTAL_T
print(f"Coincidencias adicionales por empate resueltas: {empates_extra}")
display(carga_cercana)

**Cómo leer la salida.** La tabla muestra a dónde irían los proveedores si solo importara la cercanía. Un candidato con
poco volumen cercano puede seguir siendo competitivo en el modelo económico por su costo fijo
o su cercanía a la planta. Si alguna utilización superara 100%, haría falta reasignar material:
este bloque identifica el problema, pero no resuelve una optimización con capacidad.

## 6. Comparar un almacén con el abastecimiento directo

### Un modelo de costos sencillo y completo dentro de su alcance

Para cada candidato, todo el volumen mensual sigue dos tramos: **proveedor → almacén → planta**.
Calculamos ambos. Suponer que el recorrido termina al llegar al almacén daría un ahorro artificial.

Denotamos por $q_i$ las toneladas del proveedor $i$, por $d_{iP}$ sus kilómetros rectos a planta,
por $d_{ij}$ sus kilómetros al almacén $j$, y por $d_{jP}$ la distancia del almacén a la planta.
$f$ es el factor de recorrido, $r_D$, $r_R$ y $r_C$ son las tarifas directa, de recolección y consolidada;
$h$ es el manejo por tonelada y $F_j$ el costo fijo mensual del candidato.

$$C_{directo}=\sum_i q_i\,d_{iP}\,f\,r_D$$
$$C_j=\underbrace{\sum_i q_i\,d_{ij}\,f\,r_R}_{proveedores\ a\ almacén}
    +\underbrace{(\sum_i q_i)\,d_{jP}\,f\,r_C}_{almacén\ a\ planta}
    +\underbrace{(\sum_i q_i)h}_{manejo}+\underbrace{F_j}_{costo\ fijo}$$

Las unidades se cancelan así: toneladas × kilómetros × MXN/(tonelada × kilómetro) = MXN.
Un ahorro positivo es $C_{directo}-C_j$; uno negativo significa que el almacén aumenta el costo.
Solo es factible una alternativa cuya capacidad mensual sea al menos el total de toneladas.
Mantener la operación directa siempre participa como alternativa; no forzamos abrir un almacén.

Estas tarifas y costos son inventados. El factor 1.25 no transforma la distancia en una ruta real:
solo crea una aproximación comparable. El supuesto de una tarifa consolidada exige verificar
ocupación de vehículos y frecuencia de envíos antes de aplicarlo a una fábrica real.

### Bloque 12 · Construir la función de evaluación económica

`evaluar_red` encapsula el cálculo para repetirlo con distintos factores de recorrido y tarifas.
Primero calcula la operación directa, que sirve como referencia común. Después recorre los tres
candidatos con `itertuples`, mide distancias y suma los cuatro componentes del costo de almacén.

Por cada candidato crea un diccionario con costos, ahorro, distancia ponderada de entrada,
distancia del tramo a planta, capacidad y exposición del propio sitio. `factible` revisa únicamente
capacidad; la exposición se reporta para discusión y no se monetiza ni funciona como restricción.
Si la política de la empresa prohibiera ubicaciones expuestas, habría que añadir esa restricción.
La función devuelve la referencia directa y una tabla; definirla todavía no ejecuta el análisis.

In [ ]:
def evaluar_red(factor=FACTOR_RECORRIDO, tarifa_consolidada=TARIFA_CONSOLIDADA):
    assert factor >= 1 and tarifa_consolidada >= 0
    volumen = proveedores["toneladas_mes"]
    costo_directo = float((volumen * proveedores["distancia_planta_km"] * factor * TARIFA_DIRECTA).sum())
    filas = []
    for sitio in candidatos.itertuples():
        distancias_entrada = proveedores.geometry.distance(sitio.geometry) / 1000
        distancia_salida = sitio.geometry.distance(punto_planta) / 1000
        recoleccion = float((volumen * distancias_entrada * factor * TARIFA_RECOLECCION).sum())
        consolidado = TOTAL_T * distancia_salida * factor * tarifa_consolidada
        manejo = TOTAL_T * MANEJO_MXN_T
        total = recoleccion + consolidado + manejo + sitio.fijo_mxn_mes
        filas.append({
            "candidato": sitio.candidato,
            "entrada_km_ponderada": float(np.average(distancias_entrada, weights=volumen)),
            "salida_km": distancia_salida,
            "recoleccion_mxn": recoleccion, "consolidado_mxn": consolidado,
            "manejo_mxn": manejo, "fijo_mxn": sitio.fijo_mxn_mes,
            "costo_total_mxn": total, "ahorro_mxn": costo_directo - total,
            "ahorro_pct": 100 * (costo_directo - total) / costo_directo,
            "capacidad_t_mes": sitio.capacidad_t_mes,
            "utilizacion_pct": 100 * TOTAL_T / sitio.capacidad_t_mes,
            "factible": TOTAL_T <= sitio.capacidad_t_mes,
            "sitio_expuesto": sitio.geometry.intersects(zona.geometry.iloc[0]),
        })
    return costo_directo, pd.DataFrame(filas)

print("Función definida: compara todo el volumen por un solo candidato a la vez.")

**Cómo leer la salida.** El mensaje confirma que la función está lista. La separación entre definir y llamar una función
permite reutilizar la misma lógica en la comparación base y en la sensibilidad, sin copiar fórmulas
que después podrían divergir.

### Bloque 13 · Calcular alternativas y seleccionar la de menor costo factible

Invocamos la función con los supuestos base. La primera tabla muestra componentes de costo y la
segunda, restricciones e indicadores. Filtramos candidatos con capacidad suficiente y seleccionamos
el de menor costo; después lo comparamos con seguir entregando directamente.

`mejor_candidato` representa el almacén factible menos costoso; `decision` representa la alternativa
más barata incluyendo operación directa. No son necesariamente lo mismo. Un ahorro pequeño o
una ubicación expuesta requiere una revisión operativa aun cuando el cálculo sea favorable.
El texto usa variables calculadas para mantenerse consistente cuando modifique los supuestos.

In [ ]:
costo_directo, evaluacion = evaluar_red()
componentes = ["recoleccion_mxn", "consolidado_mxn", "manejo_mxn", "fijo_mxn"]
display(evaluacion[["candidato", *componentes, "costo_total_mxn", "ahorro_mxn"]])
display(evaluacion[["candidato", "entrada_km_ponderada", "salida_km", "utilizacion_pct", "factible", "sitio_expuesto"]])
factibles = evaluacion.loc[evaluacion["factible"]].sort_values(["costo_total_mxn", "candidato"])
mejor_candidato = None if factibles.empty else factibles.iloc[0]
decision = "Directo"
costo_elegido = costo_directo
if mejor_candidato is not None and mejor_candidato["costo_total_mxn"] < costo_directo:
    decision = str(mejor_candidato["candidato"])
    costo_elegido = float(mejor_candidato["costo_total_mxn"])
ahorro_elegido = costo_directo - costo_elegido
display(Markdown(f"""
**Comparación mensual simulada:** abastecimiento directo = **{costo_directo:,.0f} MXN**.
La alternativa de menor costo factible es **{decision}**, con **{costo_elegido:,.0f} MXN/mes**.
La diferencia frente a directo es **{ahorro_elegido:,.0f} MXN/mes ({100*ahorro_elegido/costo_directo:.1f}%)**.
La selección considera costos modelados y capacidad; todavía no garantiza tiempos de entrega,
disponibilidad del inmueble ni viabilidad real de consolidación.
"""))

**Cómo leer la salida.** Una utilización superior a 100% descarta el candidato para recibir todo el flujo, aunque su costo
parezca atractivo. `sitio_expuesto=True` señala una exposición adicional a revisar. Si gana
“Directo”, el modelo indica que ninguno de los almacenes factibles mejora la referencia con
estos supuestos; eso también es un resultado útil.

### Bloque 14 · Visualizar qué componentes explican los costos

Creamos barras apiladas para distinguir recolección, transporte consolidado, manejo y fijo.
La barra directa solo tiene transporte directo. `bottom` acumula alturas para colocar cada
componente sobre el anterior; no modifica los valores económicos. Las alternativas sin capacidad
se identifican en su etiqueta y con un tramado, para que una barra baja no se interprete como una opción viable.

El eje comienza en cero y se expresa en miles de MXN por mes. `axhline` coloca la referencia directa
para comparar visualmente. La imagen exportada permite revisar este resultado sin ejecutar Folium.

In [ ]:
tabla_costos = evaluacion.set_index("candidato")[componentes].copy()
tabla_costos["directo_mxn"] = 0.0
tabla_costos.loc["Directo"] = [0, 0, 0, 0, costo_directo]
tabla_costos = tabla_costos.reindex(["Directo", *evaluacion["candidato"]])
colores = ["#2b7094", "#55a89c", "#eab45b", "#a4acb7", "#26384c"]
etiquetas = ["Recolección", "Consolidado a planta", "Manejo", "Fijo", "Transporte directo"]
fig, ax = plt.subplots(figsize=(11, 5.5))
base = np.zeros(len(tabla_costos))
for columna, color, etiqueta in zip(tabla_costos.columns, colores, etiquetas):
    valores = tabla_costos[columna].to_numpy() / 1000
    ax.bar(tabla_costos.index, valores, bottom=base, color=color, label=etiqueta)
    base += valores
no_factibles = set(evaluacion.loc[~evaluacion["factible"], "candidato"])
for posicion, nombre in enumerate(tabla_costos.index):
    ax.text(posicion, base[posicion] + 4, f"{base[posicion]:,.1f}", ha="center", fontsize=9)
    if nombre in no_factibles:
        for contenedor in ax.containers:
            contenedor[posicion].set_hatch("///")
ax.set_xticks(range(len(tabla_costos)), [n + ("\nSin capacidad" if n in no_factibles else "") for n in tabla_costos.index])
ax.axhline(costo_directo / 1000, linestyle="--", color="#26384c", alpha=0.5)
ax.set(title="Costo mensual por alternativa · Supuestos base ficticios", ylabel="Miles de MXN/mes", ylim=(0, max(base) * 1.2))
ax.legend(ncol=3, loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(SALIDA / "02_comparacion_costos.png", bbox_inches="tight")
plt.show()

**Cómo leer la salida.** Una reducción de recolección puede verse compensada por el tramo almacén–planta y el manejo.
La composición de cada barra explica esa compensación. La línea discontinua representa la operación
actual y facilita reconocer aumentos o reducciones. El tramado indica insuficiencia de capacidad,
no incertidumbre estadística.

### Bloque 15 · Probar sensibilidad: ¿cambia la decisión?

Evaluamos nueve combinaciones: tres factores de recorrido y tres tarifas consolidadas.
En cada combinación recalculamos también el costo directo para que ambas alternativas compartan
el mismo supuesto de recorrido. Se mantienen toneladas, costos fijos, capacidad y demás tarifas.

Incluimos “Directo” entre las opciones y elegimos la de menor costo factible con `min`.
Las tablas pivote muestran la decisión y su ahorro porcentual. Los nueve escenarios tienen el
mismo peso de presentación, pero no son una distribución de probabilidades: ganar seis de nueve
no implica una probabilidad de éxito de dos tercios.

In [ ]:
filas_sensibilidad = []
for factor in [1.0, 1.25, 1.5]:
    for tarifa in [1.0, 1.6, 2.5]:
        referencia, escenarios = evaluar_red(factor, tarifa)
        opciones = {"Directo": referencia}
        opciones.update(escenarios.loc[escenarios["factible"]].set_index("candidato")["costo_total_mxn"].to_dict())
        elegida = min(opciones, key=lambda nombre: (opciones[nombre], nombre))
        filas_sensibilidad.append({
            "factor_recorrido": factor, "tarifa_consolidada": tarifa,
            "decision": elegida, "costo_mxn": opciones[elegida],
            "ahorro_pct": 100 * (referencia - opciones[elegida]) / referencia,
        })
sensibilidad = pd.DataFrame(filas_sensibilidad)
display(sensibilidad.pivot(index="factor_recorrido", columns="tarifa_consolidada", values="decision"))
display(sensibilidad.pivot(index="factor_recorrido", columns="tarifa_consolidada", values="ahorro_pct").round(1))
coincidencias = int((sensibilidad["decision"] == decision).sum())
display(Markdown(f"""La decisión base **{decision}** se mantiene en **{coincidencias} de 9 escenarios**.
El ahorro de la mejor alternativa de cada escenario varía entre **{sensibilidad['ahorro_pct'].min():.1f}%**
y **{sensibilidad['ahorro_pct'].max():.1f}%**. Esto mide sensibilidad al rango elegido, no certidumbre.
Si cambia la elección, conviene precisar tarifas y recorridos antes de priorizar un candidato."""))

**Cómo leer la salida.** Lea filas como factores de recorrido y columnas como tarifas consolidadas en MXN/t-km.
La primera tabla responde quién gana; la segunda, cuánto mejora frente a directo en ese mismo escenario.
Una elección estable frente a estas dos variables todavía puede cambiar con demanda, capacidad,
restricciones de riesgo o costos de inventario, que aquí se mantienen fijos.

## 7. Explorar el mapa interactivo
### Bloque 16 · Crear capas y ventanas de información

Volvemos a EPSG:4326 para publicar las geometrías como GeoJSON. Folium usa `[latitud, longitud]`
para el centro y los marcadores, mientras GeoJSON mantiene `[longitud, latitud]` internamente;
GeoPandas se encarga de esta última conversión. El mapa de fondo proviene de OpenStreetMap.

`FeatureGroup` organiza los proveedores en una capa que puede activarse. Cada `CircleMarker`
incluye toneladas, distancia y retrasos en una ventana al pulsarlo. Agregamos la zona de riesgo,
el radio de cobertura, la planta y los candidatos. `LayerControl` ofrece interruptores y
`fit_bounds` ajusta la vista a la red. `save` exporta un HTML interactivo que necesita Internet
para cargar las librerías web y el fondo, pero no requiere un servidor propio.

In [ ]:
proveedores_mapa = proveedores.to_crs(CRS_GEOGRAFICO)
mapa = folium.Map(location=[25.80, -100.25], zoom_start=10, tiles="OpenStreetMap", control_scale=True)
folium.GeoJson(zona.to_crs(CRS_GEOGRAFICO).to_json(), name="Zona de riesgo ficticia",
    style_function=lambda _: {"color": "#c83f49", "fillOpacity": 0.18},
    tooltip="Zona inventada: no es información oficial de amenazas").add_to(mapa)
folium.GeoJson(cobertura.to_crs(CRS_GEOGRAFICO).to_json(), name="Radio recto de 25 km",
    style_function=lambda _: {"color": "#218c74", "fillOpacity": 0.03, "dashArray": "6 5"}).add_to(mapa)
capa_proveedores = folium.FeatureGroup(name="Proveedores: rojo expuesto, azul fuera")
for fila in proveedores_mapa.itertuples():
    texto = (f"<b>{fila.proveedor} · Ficticio</b><br>{fila.toneladas_mes} t/mes"
             f"<br>Distancia recta: {fila.distancia_planta_km:.1f} km"
             f"<br>Retrasos: {fila.retrasos}/{fila.entregas} ({fila.tasa_retraso_pct:.1f}%)")
    color = "#c83f49" if fila.expuesto else "#246a95"
    folium.CircleMarker([fila.geometry.y, fila.geometry.x], radius=6, color=color,
        fill=True, fill_opacity=0.85, tooltip=fila.proveedor,
        popup=folium.Popup(texto, max_width=280)).add_to(capa_proveedores)
capa_proveedores.add_to(mapa)
folium.Marker([planta_geo.geometry.iloc[0].y, planta_geo.geometry.iloc[0].x],
    tooltip="Planta MetalNorte ficticia", icon=folium.Icon(color="black", icon="home")).add_to(mapa)
for fila in candidatos_geo.itertuples():
    folium.Marker([fila.geometry.y, fila.geometry.x], tooltip=f"{fila.candidato} · Candidato ficticio",
        popup=f"Capacidad: {fila.capacidad_t_mes} t/mes", icon=folium.Icon(color="orange", icon="info-sign")).add_to(mapa)
limites = proveedores_mapa.total_bounds
mapa.fit_bounds([[limites[1] - 0.05, limites[0] - 0.05], [limites[3] + 0.05, limites[2] + 0.05]])
folium.LayerControl(collapsed=False).add_to(mapa)
mapa.save(str(SALIDA / "mapa_interactivo.html"))
display(mapa)

**Cómo leer la salida.** Active y desactive la zona ficticia y pulse varios proveedores dentro y fuera del radio.
El fondo geográfico ayuda a orientarse, pero no convierte los puntos ficticios en instalaciones
reales ni certifica que una ruta sea transitable. Los símbolos naranjas son candidatos evaluados,
no almacenes abiertos. La información comercial procede exclusivamente de nuestra simulación.

## 8. Comprobaciones del análisis
### Bloque 17 · Reconciliar cálculos importantes

Comprobamos tres riesgos comunes: perder o duplicar volumen, comparar costos incompletos y medir
distancias con unidades incorrectas. La suma de grupos y de asignaciones debe recuperar el total.
Los costos de cada almacén deben ser iguales a la suma de sus cuatro componentes.

Para una distancia individual calculamos otra referencia con `pyproj.Geod`, que mide sobre el
elipsoide terrestre a partir de longitud y latitud. Esperamos una diferencia relativa inferior
a 0.2% frente a UTM para este caso local. Esa concordancia valida escala y proyección, no carreteras.
También recomputamos el costo directo proveedor por proveedor y verificamos la elección base
contra la fila correspondiente de sensibilidad. Si un supuesto se modifica fuera de ese rango,
actualice también la combinación de sensibilidad usada en esta comprobación.

In [ ]:
from pyproj import Geod

assert proveedores["toneladas_mes"].sum() == TOTAL_T
assert resumen_riesgo["toneladas"].sum() == TOTAL_T
assert asignacion["toneladas_mes"].sum() == TOTAL_T
assert proveedores["retrasos"].between(0, proveedores["entregas"]).all()
assert np.allclose(evaluacion[componentes].sum(axis=1), evaluacion["costo_total_mxn"])
recalculo_directo = sum(float(f.toneladas_mes) * float(f.distancia_planta_km)
                       * FACTOR_RECORRIDO * TARIFA_DIRECTA for f in proveedores.itertuples())
assert np.isclose(recalculo_directo, costo_directo)
geod = Geod(ellps="WGS84")
ejemplo = proveedores.iloc[0]
_, _, distancia_geodesica_m = geod.inv(float(ejemplo["longitud"]), float(ejemplo["latitud"]),
                                      float(planta_df.loc[0, "longitud"]), float(planta_df.loc[0, "latitud"]))
error_relativo = abs(ejemplo["distancia_planta_km"] * 1000 - distancia_geodesica_m) / distancia_geodesica_m
assert error_relativo < 0.002
fila_base = sensibilidad.loc[np.isclose(sensibilidad["factor_recorrido"], FACTOR_RECORRIDO)
                            & np.isclose(sensibilidad["tarifa_consolidada"], TARIFA_CONSOLIDADA)]
if not fila_base.empty:
    assert fila_base.iloc[0]["decision"] == decision
    assert np.isclose(fila_base.iloc[0]["costo_mxn"], costo_elegido)
assert costo_elegido <= costo_directo + 1e-8
print("Validación correcta: volumen, tasas, costos, selección y escala de distancias.")
print(f"Diferencia UTM frente a distancia geodésica para P001: {100*error_relativo:.4f}%.")

**Cómo leer la salida.** Una ejecución sin errores confirma coherencia interna de las comprobaciones incluidas.
No valida las tarifas ficticias frente al mercado ni demuestra que el ahorro ocurriría en una planta real.
Esa diferencia entre exactitud del cálculo y validez de los supuestos es central para usar responsablemente el análisis.

## 9. Interpretación integrada y conclusiones
### Bloque 18 · Escribir conclusiones con resultados calculados

Este bloque prepara un texto de lectura ejecutiva con números tomados de las tablas anteriores.
Usa condicionales para distinguir una recomendación de almacén de una decisión de conservar directo,
y enumera las alternativas descartadas por capacidad. Los textos explican qué significa cada resultado
y hasta dónde permite concluir, en lugar de presentar porcentajes aislados.

El documento se guarda en Markdown y también se muestra en el notebook. Al volver a ejecutar todo,
las conclusiones cambian junto con los resultados: no quedan cifras pegadas manualmente que puedan
contradecir las tablas. La recomendación es solo una prioridad de evaluación dentro del caso ficticio.

In [ ]:
descartados = ", ".join(evaluacion.loc[~evaluacion["factible"], "candidato"]) or "Ninguno"
tasas_texto = "; ".join(f"{grupo}: {fila.tasa_retraso_pct:.1f}% ({int(fila.retrasos)}/{int(fila.entregas)} entregas)"
                       for grupo, fila in resumen_riesgo.iterrows())
accion = (f"Priorizar un estudio operativo de **{decision}**, contrastando recorridos y tarifas reales."
          if decision != "Directo" else
          "Conservar la operación directa como referencia y revisar supuestos antes de justificar un almacén.")
conclusiones = f"""
## Conclusiones del caso ficticio MetalNorte

**1. La ubicación agrega información que la tabla de compras no muestra sola.** La red tiene
60 proveedores y **{TOTAL_T:,.0f} toneladas mensuales**. Su distancia ponderada a planta es
**{distancia_ponderada:.1f} km rectos**: representa la separación media de una tonelada de abastecimiento.
No representa cuánto tarda un camión ni cuántos kilómetros recorre efectivamente.

**2. La cobertura debe leerse con el volumen.** Dentro de **{RADIO_KM:g} km** quedan
**{cobertura_proveedores:.1f}% de proveedores** y **{cobertura_toneladas:.1f}% de toneladas**.
El resto del flujo proviene de ubicaciones más lejanas según esta regla de proximidad.
Esto permite focalizar una revisión logística, sin etiquetar automáticamente como deficientes a los proveedores lejanos.

**3. La exposición territorial puede concentrarse.** La zona inventada contiene
**{expuestos_n} proveedores** y **{100*expuestos_t/TOTAL_T:.1f}% del volumen mensual**.
Esa concentración sugiere revisar planes de continuidad y alternativas de suministro si se observara
en datos reales. No equivale a una probabilidad de desastre ni considera rutas que cruzan la zona.

**4. Los retrasos ilustran una comparación descriptiva.** {tasas_texto}.
La tasa global es **{tasa_global:.1f}%**. Calculamos tasas sobre entregas, sin promediar porcentajes
de proveedores con distintos tamaños. Las diferencias se generaron parcialmente por diseño;
no se puede afirmar que un almacén las reduciría ni que la ubicación sea su única causa.

**5. La selección requiere sumar ambos tramos y respetar la capacidad.** La operación directa
cuesta **{costo_directo:,.0f} MXN/mes** en el modelo. La alternativa factible de menor costo es
**{decision}**, con **{costo_elegido:,.0f} MXN/mes**. La diferencia es **{ahorro_elegido:,.0f} MXN/mes
({100*ahorro_elegido/costo_directo:.1f}%)**. Candidatos sin capacidad para todo el volumen: **{descartados}**.
Un menor costo es una señal para investigar; no incorpora inversión inicial, inventario ni tiempos de servicio.

**6. La decisión depende de supuestos verificables.** La alternativa base coincide con la elegida
en **{coincidencias} de 9 combinaciones** de recorrido y tarifa. Esa cantidad describe estabilidad
dentro del rango ensayado, no probabilidad de éxito. Fuera de ese rango o con otras restricciones
la elección podría cambiar. **Siguiente acción dentro del caso:** {accion}
"""
display(Markdown(conclusiones))
(SALIDA / "conclusiones.md").write_text(conclusiones, encoding="utf-8")

**Cómo leer la salida.** Las conclusiones conectan hallazgos con decisiones y sus límites. Distinga un hecho calculado
—por ejemplo, toneladas dentro del polígono— de una acción propuesta —revisar continuidad— y de
un resultado que aún no se puede asegurar —reducir retrasos—. Esa separación mejora la comunicación con operaciones.

### Conclusiones generales sobre la técnica

El análisis geoespacial es útil cuando **la posición cambia la interpretación de los datos**.
En manufactura puede revelar concentración de proveedores, cobertura insuficiente de almacenes,
dependencia de corredores logísticos o proximidad entre plantas y mercados. El mapa es una forma
de explorar; el valor analítico aparece cuando sus geometrías se convierten en medidas y comparaciones.

Un CRS correcto importa tanto como una fórmula correcta. Medir en grados y llamar kilómetros al
resultado puede producir una decisión equivocada aunque todo el código corra sin errores.
Igualmente, contar puntos y medir toneladas son perspectivas complementarias: una zona con pocos
proveedores puede ser muy relevante si concentra gran parte del material.

La cercanía física ayuda a plantear opciones, pero la decisión operativa necesita capacidad,
costos, tiempos y riesgo. Nuestro ejercicio compara solo tres ubicaciones propuestas; **no busca
la ubicación óptima entre todos los puntos posibles del territorio**. La evaluación económica
es una enumeración de alternativas, no una optimización completa de red logística.

### Cómo llevarlo a una fábrica real

1. Sustituir los datos sintéticos por un maestro de proveedores con coordenadas verificadas y toneladas del mismo periodo.
2. Reemplazar el rectángulo por polígonos de amenaza o de interrupción de una fuente oficial, con fecha y metodología.
3. Incorporar distancias y tiempos por una red vial, restricciones de carga y horarios de acceso.
4. Validar tarifas, vehículos, frecuencias, inventario de seguridad, tiempos de manejo y capacidad efectiva por turno.
5. Definir si la exposición es una restricción, un criterio cualitativo o un costo esperado; no mezclar esas funciones sin explicación.
6. Comparar un piloto con una referencia operativa antes de afirmar mejoras de costo o nivel de servicio.

### Ejercicios para ampliar el aprendizaje

- Cambie `RADIO_KM` de 25 a 20 y 35; ejecute todo y explique por qué crece o disminuye la cobertura de toneladas.
- Duplique `MANEJO_MXN_T` y observe si el ahorro sobrevive. Identifique qué componente de la barra aumenta.
- Aumente la capacidad de C_Oeste y vuelva a comparar: ¿la restricción era realmente la razón por la que no ganaba?
- Cambie la semilla y observe si la jerarquía de candidatos depende de la distribución espacial simulada.
- Proponga qué dato adicional necesitaría para convertir un radio de kilómetros en una cobertura de 45 minutos.

### Errores frecuentes y cómo resolverlos

| Síntoma | Causa probable | Acción |
|---|---|---|
| No se encuentra una variable | Se ejecutaron celdas fuera de orden | Reiniciar y ejecutar todas |
| GeoPandas advierte sobre CRS geográfico | Se midieron geometrías en grados | Aplicar `to_crs(CRS_METRICO)` antes de medir |
| El mapa aparece en otra región | Se intercambió latitud y longitud | GeoPandas usa x=longitud; Folium recibe latitud primero |
| Las toneladas aumentan después de un cruce | Un punto coincide con varias zonas o vecinos empatados | Revisar duplicados y definir una regla de asignación |
| El HTML no muestra el fondo | No cargan recursos web o mosaicos | Habilitar conexión o utilizar los PNG guardados |
| Una librería importada no cambia de versión | La sesión mantiene módulos anteriores | Reiniciar el entorno después de instalar y ejecutar todas |

## 10. Guardar y descargar
### Bloque 19 · Exportar datos, métricas y un ZIP

Los CSV conservan las tablas para revisarlas en una hoja de cálculo. El GeoJSON guarda geometrías
en EPSG:4326 para usarlas en otro sistema de información geográfica. Antes de escribir CSV eliminamos
`geometry`, pues un objeto espacial no es una columna tabular ordinaria; latitud y longitud permanecen.

El manifiesto JSON registra origen sintético, periodo, semilla, CRS, parámetros y versiones.
`shutil.make_archive` reúne la carpeta de salidas en un ZIP que queda fuera de ella para evitar
incluirse a sí mismo. No se conecta a Drive ni publica archivos. Volver a ejecutar actualiza los
archivos de este tutorial con los resultados actuales.

In [ ]:
proveedores.drop(columns="geometry").to_csv(SALIDA / "proveedores_resultados.csv", index=False)
asignacion.drop(columns="geometry").to_csv(SALIDA / "asignacion_cercania.csv", index=False)
evaluacion.to_csv(SALIDA / "evaluacion_almacenes.csv", index=False)
sensibilidad.to_csv(SALIDA / "sensibilidad.csv", index=False)
resumen_riesgo.to_csv(SALIDA / "resumen_exposicion.csv")
for nombre, capa in [("proveedores", proveedores), ("candidatos", candidatos), ("zona_ficticia", zona), ("cobertura", cobertura)]:
    (SALIDA / f"{nombre}.geojson").write_text(capa.to_crs(CRS_GEOGRAFICO).to_json(), encoding="utf-8")
manifiesto = {
    "origen": "Datos completamente sintéticos; caso didáctico MetalNorte",
    "periodo": PERIODO, "semilla": SEMILLA, "crs_medicion": CRS_METRICO,
    "versiones": versiones, "python": sys.version,
    "supuestos": {"radio_km": RADIO_KM, "factor_recorrido": FACTOR_RECORRIDO,
                  "tarifa_directa_mxn_tkm": TARIFA_DIRECTA, "tarifa_recoleccion_mxn_tkm": TARIFA_RECOLECCION,
                  "tarifa_consolidada_mxn_tkm": TARIFA_CONSOLIDADA, "manejo_mxn_t": MANEJO_MXN_T},
    "resultado": {"toneladas": TOTAL_T, "decision": decision, "costo_directo_mxn": costo_directo,
                  "costo_elegido_mxn": costo_elegido, "ahorro_mxn": ahorro_elegido},
}
(SALIDA / "manifiesto.json").write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding="utf-8")
archivo_zip = shutil.make_archive(str(SALIDA.parent / "resultados_geoespacial_manufactura"), "zip", root_dir=SALIDA)
print("ZIP preparado:", Path(archivo_zip).name)
display(pd.DataFrame({"Archivo": sorted(p.name for p in SALIDA.iterdir() if p.is_file())}))

**Cómo leer la salida.** El ZIP contiene datos de entrada y resultados, dos figuras PNG, el mapa HTML, geometrías GeoJSON,
conclusiones y un manifiesto. El notebook se conserva como archivo separado: descárguelo desde
el menú Archivo de Colab si desea guardar también las celdas y sus salidas.

### Bloque 20 · Descargar los resultados desde Colab

`google.colab.files.download` solicita al navegador la descarga del ZIP ya creado.
El `try/except` permite ejecutar el mismo notebook fuera de Colab: si su módulo no existe,
mostramos la ubicación local del archivo. La descarga puede requerir que el navegador permita
descargas; también puede hacerlo manualmente desde el panel Archivos de Colab.

In [ ]:
try:
    from google.colab import files
except ImportError:
    print("Fuera de Colab. Archivo disponible en:", Path(archivo_zip).resolve())
else:
    files.download(archivo_zip)

**Cómo leer la salida.** Si está en Colab, debe comenzar la descarga de `resultados_geoespacial_manufactura.zip`.
Si ejecuta localmente, la ruta impresa indica dónde se guardó. No es necesario subir información
corporativa para completar este tutorial.